In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
from osgeo import gdal
import os
import numpy as np
import cv2
from scipy import ndimage
import radtran as rt
from glob import glob
from mpl_toolkits.axes_grid1 import make_axes_locatable
from skimage import filters

In [ ]:
# Normalized Difference Vegetation Index (NDVI)
x_s, x_e, y_s, y_e = (
    1100,
    1300,
    800,
    1000,
)  # 0,1026,0,1016 #250,750,250,750 #250,750,250,750 #400,600,400,600 #0,996,0,1000 #250,750,250,750 #0,1000,0,1000 #500,700,400,600 #350,650,350,650 #400,600,400,600  #300,700,300,700


# [400:600,400:600]  4km*4km  500,700,400,600
# [250:750,250:750]  10km*10km
def NDVI(data):
    band8 = data.GetRasterBand(8).ReadAsArray()[x_s:x_e, y_s:y_e]
    band4 = data.GetRasterBand(4).ReadAsArray()[x_s:x_e, y_s:y_e]
    return (band8 - band4) / (band8 + band4)


def NDWI(data):
    band3 = data.GetRasterBand(3).ReadAsArray()[x_s:x_e, y_s:y_e]
    band5 = data.GetRasterBand(5).ReadAsArray()[x_s:x_e, y_s:y_e]
    return (band3 - band5) / (band3 + band5)


# Normalized Difference Built-up Index (NDBI)
def NDBI(data):
    band11 = data.GetRasterBand(12).ReadAsArray()[x_s:x_e, y_s:y_e]
    band8 = data.GetRasterBand(8).ReadAsArray()[x_s:x_e, y_s:y_e]
    return (band11 - band8) / (band11 + band8)


# Bare Soil Index (BSI)
def BSI(data):
    band11 = data.GetRasterBand(12).ReadAsArray()[x_s:x_e, y_s:y_e]
    band8 = data.GetRasterBand(8).ReadAsArray()[x_s:x_e, y_s:y_e]
    band4 = data.GetRasterBand(4).ReadAsArray()[x_s:x_e, y_s:y_e]
    band2 = data.GetRasterBand(2).ReadAsArray()[x_s:x_e, y_s:y_e]
    return ((band11 + band4) - (band8 + band2)) / ((band11 + band4) + (band8 + band2))


# Normalized Difference Methane Index (NDMI)
def NDMI(data):
    band12 = data.GetRasterBand(13).ReadAsArray()[x_s:x_e, y_s:y_e]
    band11 = data.GetRasterBand(12).ReadAsArray()[x_s:x_e, y_s:y_e]
    return np.abs((band12 - band11) / (band11 + band12))


# RGB
def nor_img(channel_data):
    # 将 uint16 转换为浮点数，并归一化到 [0, 1] 范围内
    min_value = channel_data.min()
    max_value = channel_data.max()
    normalized_data = (channel_data - min_value) / (max_value - min_value)
    return normalized_data


def RGB_img(data):
    channel_data2 = nor_img(data.GetRasterBand(4).ReadAsArray()[x_s:x_e, y_s:y_e])
    channel_data3 = nor_img(data.GetRasterBand(3).ReadAsArray()[x_s:x_e, y_s:y_e])
    channel_data4 = nor_img(data.GetRasterBand(2).ReadAsArray()[x_s:x_e, y_s:y_e])
    channel_data = np.dstack((channel_data2, channel_data3, channel_data4))
    return channel_data


def RGB_img_v2(data):
    channel_data2 = nor_img(data.GetRasterBand(4).ReadAsArray())
    channel_data3 = nor_img(data.GetRasterBand(3).ReadAsArray())
    channel_data4 = nor_img(data.GetRasterBand(2).ReadAsArray())
    channel_data = np.dstack((channel_data2, channel_data3, channel_data4))
    return channel_data


# align band11 and band12 (USEFUL) = histogram matching
def band_ratio_w_hm(data):
    band12 = data.GetRasterBand(13).ReadAsArray()[x_s:x_e, y_s:y_e]
    band11 = data.GetRasterBand(12).ReadAsArray()[x_s:x_e, y_s:y_e]
    c = band11.mean() / band12.mean()
    if band11.min() == 0:
        band11 += 1
    if band12.min() == 0:
        band12 += 1
    return np.log(c * band12 / band11)


# using BCET
def band_ratio_w_BCET(data):
    band12 = data.GetRasterBand(13).ReadAsArray()[x_s:x_e, y_s:y_e]
    band11 = data.GetRasterBand(12).ReadAsArray()[x_s:x_e, y_s:y_e]
    # band_data = double(band_data) #Convert the input image from 8-bit unsigned integer format to double format for processing
    Imin, Imax, Imean, Imssum = (
        np.min(band12),
        np.max(band12),
        np.mean(band12),
        np.mean(band12**2),
    )
    # Omin, Omax, Omean = 0, 255, 120
    Omin, Omax, Omean = np.min(band11), np.max(band11), np.mean(band11)
    bnum = Imax**2 * (Omean - Omin) - Imssum * (Omax - Omin) + Imin**2 * (Omax - Omean)
    bden = 2 * (Imax * (Omean - Omin) - Imean * (Omax - Omin) + Imin * (Omax - Omean))
    b = bnum / bden
    a = (Omax - Omin) / ((Imax - Imin) * (Imax + Imin - 2 * b))
    c = Omin - a * (Imin - b) ** 2
    band12_new = a * (band12 - b) ** 2 + c
    # res = uint8(res) #convert the output image back to the 8-bit unsigned integer format
    if band11.min() == 0:
        band11 += 1
    return np.log(band12_new / band11)


def BCET(data, index):
    band_data = data.GetRasterBand(index + 1).ReadAsArray()[x_s:x_e, y_s:y_e]
    # band_data = double(band_data) #Convert the input image from 8-bit unsigned integer format to double format for processing
    Imin, Imax, Imean, Imssum = (
        np.min(band_data),
        np.max(band_data),
        np.mean(band_data),
        np.mean(band_data**2),
    )
    Omin, Omax, Omean = 0, 8192, 4095  # 65535, 32767
    # Omin, Omax, Omean = np.min(band11),np.max(band11),np.mean(band11)
    bnum = Imax**2 * (Omean - Omin) - Imssum * (Omax - Omin) + Imin**2 * (Omax - Omean)
    bden = 2 * (Imax * (Omean - Omin) - Imean * (Omax - Omin) + Imin * (Omax - Omean))
    b = bnum / bden
    a = (Omax - Omin) / ((Imax - Imin) * (Imax + Imin - 2 * b))
    c = Omin - a * (Imin - b) ** 2
    band12_new = a * (band_data - b) ** 2 + c
    # res = uint8(res) #convert the output image back to the 8-bit unsigned integer format
    return band12_new


# band ratio new
def band_ratio_wo_hm(data):
    band12 = data.GetRasterBand(13).ReadAsArray()[x_s:x_e, y_s:y_e]
    band11 = data.GetRasterBand(12).ReadAsArray()[x_s:x_e, y_s:y_e]
    if band11.min() == 0:
        band11 += 1
    if band12.min() == 0:
        band12 += 1
    return np.log(band12 / band11)


# generate mask based on histogram
def gene_mask(data):
    image = band_ratio_wo_hm(data)
    scaled_image = (image - np.min(image)) / (np.max(image) - np.min(image)) * 255
    scaled_image = scaled_image.astype(np.uint8)
    threshold = filters.threshold_otsu(scaled_image)
    mask = scaled_image > threshold
    return mask


# return previous [num] dates
def prev_dates(images_folder, clouds, date, nums):
    names = []
    for name in os.listdir(images_folder):
        if name not in clouds:
            names.append(name)
    folders_arr = np.array(names)
    last_index = np.where(folders_arr < date)[0][-nums:]
    return folders_arr[last_index]


# return subsequent [num] dates
def subs_dates(images_folder, clouds, date, nums):
    names = []
    for name in os.listdir(images_folder):
        if name not in clouds:
            names.append(name)
    folders_arr = np.array(names)
    last_index = np.where(folders_arr > date)[0][:nums]
    return folders_arr[last_index]

In [ ]:
# IME Funcs
m_ch4 = 0.01614  # mass of methane kg/mol


def get_IME(enh_img, mask, a=25**2):
    delta_x = enh_img * mask  # enh_img: mol/m^2
    IME = np.sum(delta_x * m_ch4 * a)  # kg
    return IME


def get_ueff(u10, v10):
    # This is the approach use by varon et al but may need to recalibrate
    u10 = np.mean(u10)
    v10 = np.mean(v10)
    ueff = (u10**2 + v10**2) ** (1 / 2)
    # ueff = np.log(ueff)+0.5
    #    print('ueff = '+str(ueff))
    return ueff


def get_ueff_v2(u10, v10, ueff_slope=0.18801551, ueff_int=0.8801000582013674):
    u10 = np.mean(u10)
    v10 = np.mean(v10)
    U10 = (u10**2 + v10**2) ** (1 / 2)
    u_eff = ueff_slope * U10 + ueff_int
    return u_eff


# calculate source rate
def get_source_rate(img, mask, u10, v10, a=20**2):
    L = (np.sum(mask) * a) ** (
        1 / 2
    )  # a: pixel resolution  L:square root of the plume area
    IME = get_IME(img, mask, a)  # IME
    u_eff = get_ueff(u10, v10)  # Ueff
    Q = u_eff / L * IME / (10**3) * 3600  # t/h    #Source Rate:t/h
    return Q


# calculate source rate
def get_source_rate_v2(
    img,
    mask,
    u10,
    v10,
    size=100,
    a=20**2,
    ueff_slope=0.18801551,
    ueff_int=0.8801000582013674,
):  # 0.05,2.98 #0.18801551,0.8801000582013674
    mask = mask * center_mask(mask, size)
    L = (np.sum(mask) * a) ** (
        1 / 2
    )  # a: pixel resolution  L:square root of the plume area
    IME = get_IME(img, mask, a)  # IME
    u10 = np.mean(u10)
    v10 = np.mean(v10)
    u_eff = get_ueff(u10, v10)
    u_eff = 0.2 * np.log(u_eff) + 0.45
    # u_eff = get_ueff_v2(u10,v10,ueff_slope,ueff_int)  #Ueff
    Q = u_eff / L * IME / (10**3) * 3600  # t/h    #Source Rate:t/h
    #    print('Q = '+str(Q))
    return Q


def center_mask(whole_mask, size):
    rows, cols = whole_mask.shape
    mask = np.zeros((rows, cols))
    center_row, center_col = rows // 2, cols // 2

    start_row = max(center_row - size, 0)
    end_row = min(center_row + size, rows)
    start_col = max(center_col - size, 0)
    end_col = min(center_col + size, cols)

    mask[start_row:end_row, start_col:end_col] = 1
    return mask


def non_center_mask(whole_mask, size):
    rows, cols = whole_mask.shape
    mask = np.zeros((rows, cols))
    center_row, center_col = rows // 2, cols // 2

    start_row = max(center_row - size, 0)
    end_row = min(center_row + size, rows)
    start_col = max(center_col - size, 0)
    end_col = min(center_col + size, cols)

    mask[start_row:end_row, start_col:end_col] = whole_mask[
        start_row:end_row, start_col:end_col
    ]
    return mask

In [ ]:
def plume_quantification(
    dir_path,
    date,
    alpha,
    flag,
    solarangle,
    obsangle,
    savepath,
    num_layers=100,
    targheight=0,
    obsheight=100,
    instrument="S2A",
    method="MBMP_log",
):
    clouds = []
    # print(dir_path, date)
    dates = prev_dates(dir_path, clouds, date, nums=1)
    p_dates = subs_dates(dir_path, clouds, date, nums=1)
    data0 = gdal.Open(
        glob(dir_path + dates[0] + "/**/*.tiff")[0]
    )  # choose the previous date
    data = gdal.Open(
        glob(dir_path + date + "/**/*.tiff")[0]
    )  # choose the specific data
    data1 = gdal.Open(
        glob(dir_path + p_dates[0] + "/**/*.tiff")[0]
    )  # choose the specific data
    delta = band_ratio_wo_hm(data) - band_ratio_wo_hm(data1)
    # delta = band_ratio_wo_hm(data)-band_ratio_wo_hm(data0)
    # delta = band_ratio_wo_hm(data)-band_ratio_wo_hm(data2)

    ndvi_map = NDVI(data)
    # bsi_map = np.abs(BSI(data))
    # ndbi_map = NDBI(data)
    ndwi_map = NDWI(data)
    sobel_x = cv2.Sobel(ndvi_map, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(ndvi_map, cv2.CV_64F, 0, 1, ksize=3)
    gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    mask0 = gradient_magnitude < (gradient_magnitude.mean() + gradient_magnitude.std())
    if flag == 0:
        mask1 = ndvi_map < 0.2
    elif flag == 1:
        mask1 = (ndvi_map < 0.2) & gene_mask(data)
    else:
        mask1 = ndvi_map < 0.2
        mask1 = ~mask1

    # rgb_image = RGB_img(data)
    res = (
        (band_ratio_w_hm(data0) - band_ratio_w_hm(data))
        * mask1
        * ~(ndwi_map > 0)
        * mask0
    )
    res_mask = ((res > (res.mean() + alpha * res.std()))).astype(int)  # masks
    res1 = (
        (band_ratio_w_hm(data1) - band_ratio_w_hm(data))
        * mask1
        * ~(ndwi_map > 0)
        * mask0
    )
    res_mask1 = (res1 > (res1.mean() + alpha * res1.std())).astype(int)  # masks

    sobel_x = cv2.Sobel(RGB_img(data)[:, :, 0], cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(RGB_img(data)[:, :, 0], cv2.CV_64F, 0, 1, ksize=3)
    gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    mask_r = gradient_magnitude < (gradient_magnitude.mean() + gradient_magnitude.std())
    final_res = res_mask * res_mask1 * mask_r
    final_res = ndimage.median_filter(final_res, size=5)

    # # frac_refl_data = delta #*final_res
    frac_refl_data = delta * final_res

    # Try the column retrieval
    # test_retrieval_new_v2 = np.zeros_like(final_res)
    test_retrieval = rt.retrieve(
        frac_refl_data,
        instrument,
        method,
        targheight,
        obsheight,
        solarangle,
        obsangle,
        num_layers=num_layers,
    )
    test_retrieval_new_v2 = np.nan_to_num(test_retrieval)
    np.save(savepath + "_emission_b.npy", test_retrieval_new_v2)
    np.save(savepath + "_mask_b.npy", final_res)
    np.save(savepath + "_init.npy", res)

    return test_retrieval_new_v2, final_res


def cal_emission_rate(
    wind_file_path, time_index, tar_lat, tar_lon, test_retrieval_new_v2, final_res, size
):
    data = xr.open_dataset(wind_file_path)

    # Extract longitude, latitude, and u10 data
    lon = data["longitude"]
    lat = data["latitude"]
    v10 = data["v10"][time_index, :, :]
    u10 = data["u10"][time_index, :, :]
    # time = data['time'][time_index]

    # find the closest location
    diff_lat = [abs(x - tar_lat) for x in lat]
    lat_index = diff_lat.index(min(diff_lat))
    diff_lon = [abs(x - tar_lon) for x in lon]
    lon_index = diff_lon.index(min(diff_lon))
    er = get_source_rate_v2(
        test_retrieval_new_v2,
        final_res,
        u10[lat_index, lon_index].values,
        v10[lat_index, lon_index].values,
        size,
    )  # t/h
    return er

In [ ]:
# case2 2020-07-11+all types
# from sklearn.mixture import GaussianMixture
data0 = gdal.Open("./point1/2024-03-19/387a93b0e56740c73cd52833ed02e44b/response.tiff")
data = gdal.Open("./point1/2024-04-03/2252c076bf217992a60ec44d30dd287f/response.tiff")
data1 = gdal.Open("./point1/2024-04-08/c20483053efa1ee5680b24544ccb9b2f/response.tiff")
date_str = "2024-04-03"

alpha = 1.0  # 1.5
fig, axs = plt.subplots(3, 5, figsize=(20, 12))

ndvi_map = NDVI(data)
ndbi_map = NDBI(data)
ndwi_map = NDWI(data)


sobel_x = cv2.Sobel(ndvi_map, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(ndvi_map, cv2.CV_64F, 0, 1, ksize=3)
gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
mask0 = gradient_magnitude < (gradient_magnitude.mean() + gradient_magnitude.std())

mask1 = ndvi_map < 1  # ndvi_map<0.2
# mask1 = (ndvi_map<0.2)&gene_mask(data)
# mask1 = ~mask1

ax = axs[0, 0]
im = ax.imshow(RGB_img(data), cmap="viridis")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("RGB")

ax = axs[0, 1]
im = ax.imshow(ndvi_map, cmap="viridis")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("NDVI")

ax = axs[0, 2]
im = ax.imshow(ndwi_map > 0, cmap="gray")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("Water ponds")

ax = axs[0, 3]
im = ax.imshow(mask0, cmap="gray")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("SObel Operator")

ax = axs[0, 4]
im = ax.imshow(mask1, cmap="gray")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("Vegetation")

# No mapping
ax = axs[1, 0]
res = band_ratio_w_hm(data)
im = ax.imshow(res, cmap="RdBu_r")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("B12/B11")

# different dates(mapping)
ax = axs[1, 1]
res = band_ratio_w_hm(data0) - band_ratio_w_hm(data)
im = ax.imshow(res, cmap="RdBu_r", vmin=-0.1, vmax=0.1)
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("B12/B11-Ref")

# filter water
ax = axs[1, 2]
res = (band_ratio_w_hm(data0) - band_ratio_w_hm(data)) * ~(ndwi_map > 0)
im = ax.imshow(res, cmap="RdBu_r", vmin=-0.1, vmax=0.1)
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("R_water")

# edges
ax = axs[1, 3]
res = (band_ratio_w_hm(data0) - band_ratio_w_hm(data)) * ~(ndwi_map > 0) * mask0
im = ax.imshow(res, cmap="RdBu_r", vmin=-0.1, vmax=0.1)
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("R_road")

# green
ax = axs[1, 4]
res = (band_ratio_w_hm(data0) - band_ratio_w_hm(data)) * ~(ndwi_map > 0) * mask0 * mask1
im = ax.imshow(res, cmap="RdBu_r", vmin=-0.1, vmax=0.1)
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("Initial plume generation")

ax = axs[2, 0]
res = (band_ratio_w_hm(data0) - band_ratio_w_hm(data)) * mask1 * ~(ndwi_map > 0) * mask0
# thresh = threshold_otsu(res)
# print(res.mean()+alpha*res.std())
# # Segment based on threshold
# res_mask = res > thresh
res_mask = ((res > (res.mean() + alpha * res.std()))).astype(int)  # masks
res_mask = ndimage.median_filter(res_mask, size=5)
im = ax.imshow(res_mask, cmap="gray")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("CD1")

ax = axs[2, 1]
res1 = (
    (band_ratio_w_hm(data1) - band_ratio_w_hm(data)) * mask1 * ~(ndwi_map > 0) * mask0
)
res_mask1 = (res1 > (res1.mean() + alpha * res1.std())).astype(int)  # masks
res_mask1 = ndimage.median_filter(res_mask1, size=5)
im = ax.imshow(res_mask1, cmap="gray")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("CD2")

ax = axs[2, 2]
im = ax.imshow(res_mask * res_mask1, cmap="gray")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("Image multiplication")

# ax = axs[2,3]
# sobel_x = cv2.Sobel(RGB_img(data)[:,:,0], cv2.CV_64F, 1, 0, ksize=3)
# sobel_y = cv2.Sobel(RGB_img(data)[:,:,0], cv2.CV_64F, 0, 1, ksize=3)
# gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
# mask_r= gradient_magnitude<(gradient_magnitude.mean()+gradient_magnitude.std())
# final_res = res_mask*res_mask1 #*mask_r
# im = ax.imshow(final_res, cmap='gray')
# divider = make_axes_locatable(ax)
# cax = divider.append_axes("right", size="5%", pad=0.05)
# cbar = fig.colorbar(im, cax=cax)
# ax.set_title('Sobel operator')

# ax = axs[2,4]
# final_res1 = ndimage.median_filter(final_res, size=5)
# im = ax.imshow(final_res1, cmap='gray')
# divider = make_axes_locatable(ax)
# cax = divider.append_axes("right", size="5%", pad=0.05)
# cbar = fig.colorbar(im, cax=cax)
# ax.set_title('Final')

ax = axs[2, 3]
# normalized_img = (res - res.min()) / (res.max() - res.min())
# normalized_img = normalized_img * 255
# normalized_img = normalized_img.astype(np.uint8)
# pixels = normalized_img.flatten().reshape(-1, 1)

# n_components = 100
# gmm = GaussianMixture(n_components=n_components, random_state=0)
# gmm.fit(pixels)

# # 预测每个像素的分布类别
# labels = gmm.predict(pixels)

# # 3. 结果可视化
# # 将预测结果重新构造成图像
# final_res = labels.reshape(normalized_img.shape)

final_res = res_mask * res_mask1
final_res = ndimage.median_filter(final_res, size=5)

im = ax.imshow(final_res, cmap="gray")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("First")

ax = axs[2, 4]
res = (
    (band_ratio_w_BCET(data0) - band_ratio_w_BCET(data))
    * mask1
    * ~(ndwi_map > 0)
    * mask0
)
res_mask = ((res > (res.mean() + 2 * res.std()))).astype(int)  # masks
res1 = (
    (band_ratio_w_BCET(data1) - band_ratio_w_BCET(data))
    * mask1
    * ~(ndwi_map > 0)
    * mask0
)
res_mask1 = (res1 > (res1.mean() + 1.8 * res1.std())).astype(int)  # masks
final_res = res_mask * res_mask1
final_res = ndimage.median_filter(final_res, size=5)
im = ax.imshow(final_res, cmap="gray")
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im, cax=cax)
ax.set_title("BCET")

# 调整子图之间的间距
plt.tight_layout()

# 显示图像
plt.show()

In [ ]:
date = "2020-07-11"
lat = 31.7335
lng = -102.0421
solarangle = 20.045462  # Solar zenith angle in degrees
obsangle = 8.608562  # Viewing zenith angle of instrument in degrees
wind_file_path = "./21_data_wind/2020-07-11T.nc"
time_index = 17
subpath = "./21_data/".split("/")
image_path = "./21_data/"
size = 300
alpha = 1.5
flag = 0

savepath = "./saved_path/"
emis_image, mask_image = plume_quantification(
    image_path, date, alpha, flag, solarangle, obsangle, savepath
)
er = cal_emission_rate(
    wind_file_path, time_index, lat, lng, emis_image, mask_image, size
)
print(er)